# World Cup 2026 Simulator — Maher Attack/Defense Model

This version replaces the single-ELO scoring model with a **Maher (1982) attack/defense model** fit by maximum likelihood from 8 years of international match results.

**What changes vs the ELO-only version**:
- Each team now has two latent parameters: `attack` (α) and `defense` (β)
- Match goals are Poisson with `λ = α_attacker × β_defender × home_advantage`
- Defensive teams produce lower-scoring matches; offensive teams produce higher-scoring ones, even at equal "overall strength"
- Italy-style "score little, concede little" teams emerge naturally from the fit

**Data**:
- Source: International match results (results_clean.csv, 49,287 matches since 1872)
- Fit window: matches from June 2018 onward (~7,800 matches)
- Match weighting: exponential time decay (half-life 2 years) × importance weight (WC = 1.0, friendlies = 0.5)

**Falls back to ELO-derived priors** for teams with insufficient match history (none in this tournament — all 48 teams have 50+ matches in the window — but the code supports it).

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional
from collections import defaultdict
from scipy.optimize import minimize
import itertools

rng = np.random.default_rng(seed=42)

# Tournament uses some name variants vs the results dataset
# Map tournament names → results.csv names
TOURNAMENT_TO_DATASET = {
    'Korea Republic': 'South Korea',
    'Czechia': 'Czech Republic',
    "Côte d'Ivoire": 'Ivory Coast',
    'USA': 'United States',
    'Türkiye': 'Turkey',
}
# Reverse map for displaying back
DATASET_TO_TOURNAMENT = {v: k for k, v in TOURNAMENT_TO_DATASET.items()}

def toDatasetName(name: str) -> str:
    return TOURNAMENT_TO_DATASET.get(name, name)

def toTournamentName(name: str) -> str:
    return DATASET_TO_TOURNAMENT.get(name, name)

## 1. Load match results and apply filters

We use the 8-year window (June 2018 → present) with the data Tyler has already cleaned and normalized.

In [2]:
def loadMatchData(
    path: str = '/mnt/user-data/uploads/results_clean.csv',
    startDate: str = '2018-06-01',
) -> pd.DataFrame:
    """Load and filter international match results.
    
    Returns a DataFrame of completed matches with columns:
        date, home_team, away_team, home_score, away_score, tournament, neutral
    Future fixtures (NaN scores) are dropped.
    """
    df = pd.read_csv(path, parse_dates=['date'])
    df = df[df['date'] >= pd.Timestamp(startDate)].copy()
    df = df[df['home_score'].notna() & df['away_score'].notna()].copy()
    df['home_score'] = df['home_score'].astype(int)
    df['away_score'] = df['away_score'].astype(int)
    return df.reset_index(drop=True)


matches = loadMatchData()
print(f"Loaded {len(matches):,} completed matches since 2018-06-01")
print(f"Date range: {matches['date'].min().date()} → {matches['date'].max().date()}")
print(f"Unique teams: {len(set(matches['home_team']) | set(matches['away_team']))}")

Loaded 7,732 completed matches since 2018-06-01
Date range: 2018-06-01 → 2026-03-31
Unique teams: 281


## 2. Match weighting

Two factors multiplied together:

**Importance weight** (based on tournament type):
- Major tournaments (WC, Euros, Copa, AFCON, Asian Cup): 1.0
- Qualifiers: 0.85
- Nations Leagues: 0.75
- Friendlies and other: 0.5

**Time decay**: exponential with 2-year half-life. A match from 2 years ago counts half as much; 4 years ago, a quarter.

In [3]:
# Tournament importance weights
IMPORTANCE_WEIGHTS = {
    'FIFA World Cup': 1.0,
    'UEFA Euro': 1.0,
    'Copa América': 1.0,
    'African Cup of Nations': 1.0,
    'AFC Asian Cup': 1.0,
    'Gold Cup': 0.9,
    'Confederations Cup': 0.9,
    'FIFA World Cup qualification': 0.85,
    'UEFA Euro qualification': 0.85,
    'Copa América qualification': 0.85,
    'African Cup of Nations qualification': 0.85,
    'AFC Asian Cup qualification': 0.85,
    'Gold Cup qualification': 0.85,
    'UEFA Nations League': 0.75,
    'CONCACAF Nations League': 0.75,
    'Friendly': 0.5,
}

def importanceWeight(tournament: str) -> float:
    """Return weight for a tournament; defaults to 0.5 for unknowns (mostly friendlies/minor)."""
    if tournament in IMPORTANCE_WEIGHTS:
        return IMPORTANCE_WEIGHTS[tournament]
    tLower = tournament.lower()
    if 'qualification' in tLower or 'qualifier' in tLower:
        return 0.85
    return 0.5  # default: treat unknowns as friendly-ish


def timeDecayWeight(
    matchDate: pd.Timestamp,
    referenceDate: pd.Timestamp,
    halfLifeDays: float = 730,  # ~2 years
) -> float:
    """Exponential decay weight: weight = 0.5 ** (age_days / half_life_days)."""
    ageDays = (referenceDate - matchDate).days
    return 0.5 ** (ageDays / halfLifeDays)


def computeMatchWeights(
    matches: pd.DataFrame,
    referenceDate: Optional[pd.Timestamp] = None,
    halfLifeDays: float = 730,
) -> np.ndarray:
    """Compute (n_matches,) weight array combining importance and time decay."""
    if referenceDate is None:
        referenceDate = matches['date'].max()
    importance = matches['tournament'].map(importanceWeight).values
    decay = np.array([
        timeDecayWeight(d, referenceDate, halfLifeDays) for d in matches['date']
    ])
    return importance * decay


# Compute weights
weights = computeMatchWeights(matches)
print(f"Weight stats: min={weights.min():.4f}, mean={weights.mean():.4f}, max={weights.max():.4f}")
print(f"Effective sample size: {weights.sum():.0f} (vs {len(matches):,} raw matches)")

Weight stats: min=0.0331, mean=0.2590, max=0.9339
Effective sample size: 2003 (vs 7,732 raw matches)


## 3. Fit the Maher model

**Model**: For a match between home team H and away team A,

$$\lambda_H = \alpha_H \cdot \beta_A \cdot \gamma \quad\text{(home team's expected goals)}$$
$$\lambda_A = \alpha_A \cdot \beta_H \quad\text{(away team's expected goals)}$$

Where:
- $\alpha_i$ = team $i$'s attack strength (higher = scores more)
- $\beta_i$ = team $i$'s defense weakness (higher = concedes more — so good defenses have LOW β)
- $\gamma$ = home advantage multiplier
- Neutral venue matches drop the $\gamma$ term

Goals are Poisson:
$$P(H_{goals}=h, A_{goals}=a) = \frac{e^{-\lambda_H}\lambda_H^h}{h!} \cdot \frac{e^{-\lambda_A}\lambda_A^a}{a!}$$

We fit by **weighted maximum likelihood**:
$$\mathcal{L} = \sum_{\text{matches}} w_m \left[ \log P(h_m | \lambda_H) + \log P(a_m | \lambda_A) \right]$$

**Identifiability**: The model has redundancy (you can scale all $\alpha$ up and all $\beta$ down proportionally without changing $\lambda$). We fix this by constraining $\prod \alpha_i = 1$ (geometric mean = 1), which is the standard approach.

In [4]:
def fitMaherModel(
    matches: pd.DataFrame,
    weights: np.ndarray,
    minMatches: int = 10,
    verbose: bool = True,
) -> dict:
    """Fit Maher attack/defense parameters by weighted MLE with analytical gradient.
    
    The analytical gradient is critical: with ~450 parameters, numerical gradients
    burn through function evaluations and L-BFGS-B fails to converge in the default budget.
    With the analytical gradient, convergence happens in ~300 iterations.
    
    Returns dict with: attack, defense, homeAdv, teams, logLik, converged
    """
    homeCounts = matches['home_team'].value_counts()
    awayCounts = matches['away_team'].value_counts()
    totalCounts = homeCounts.add(awayCounts, fill_value=0).astype(int)
    eligible = sorted(totalCounts[totalCounts >= minMatches].index.tolist())
    teamToIdx = {t: i for i, t in enumerate(eligible)}
    nTeams = len(eligible)
    
    if verbose:
        print(f"Fitting Maher model on {nTeams} teams ({minMatches}+ matches)")
    
    mask = matches['home_team'].isin(teamToIdx) & matches['away_team'].isin(teamToIdx)
    fitMatches = matches[mask].copy().reset_index(drop=True)
    fitWeights = weights[mask.values]
    if verbose:
        print(f"  Matches used: {len(fitMatches):,} ({mask.sum()/len(matches)*100:.1f}% of total)")
    
    homeIdx = fitMatches['home_team'].map(teamToIdx).values
    awayIdx = fitMatches['away_team'].map(teamToIdx).values
    homeGoals = fitMatches['home_score'].values.astype(float)
    awayGoals = fitMatches['away_score'].values.astype(float)
    neutral = fitMatches['neutral'].values.astype(bool)
    nonNeutral = ~neutral
    
    # Parameter layout: [logAlpha_free (n-1), logBeta (n), logGamma (1)]
    # Constraint: sum(logAlpha) = 0 → logAlpha[0] = -sum(logAlpha_free)
    
    def negLogLikAndGrad(params):
        logAlphaFree = params[:nTeams-1]
        logAlpha0 = -logAlphaFree.sum()
        logAlpha = np.concatenate([[logAlpha0], logAlphaFree])
        logBeta = params[nTeams-1:2*nTeams-1]
        logGamma = params[2*nTeams-1]
        
        alpha = np.exp(logAlpha)
        beta = np.exp(logBeta)
        gamma = np.exp(logGamma)
        
        homeAdvFactor = np.where(neutral, 1.0, gamma)
        lambdaH = np.clip(alpha[homeIdx] * beta[awayIdx] * homeAdvFactor, 1e-6, 20)
        lambdaA = np.clip(alpha[awayIdx] * beta[homeIdx], 1e-6, 20)
        
        ll = homeGoals * np.log(lambdaH) - lambdaH + awayGoals * np.log(lambdaA) - lambdaA
        negLL = -(fitWeights * ll).sum()
        
        # Gradient (in log-parameter space):
        # d(log P)/d(logAlpha_k) = sum over matches involving k: (observed - expected)
        resH = fitWeights * (homeGoals - lambdaH)
        resA = fitWeights * (awayGoals - lambdaA)
        
        gradLogAlpha = np.zeros(nTeams)
        np.add.at(gradLogAlpha, homeIdx, resH)
        np.add.at(gradLogAlpha, awayIdx, resA)
        
        gradLogBeta = np.zeros(nTeams)
        np.add.at(gradLogBeta, awayIdx, resH)
        np.add.at(gradLogBeta, homeIdx, resA)
        
        gradLogGamma = resH[nonNeutral].sum()
        
        # Negate for minimization
        gradLogAlpha = -gradLogAlpha
        gradLogBeta = -gradLogBeta
        gradLogGamma = -gradLogGamma
        
        # Chain rule for constraint: gradient w.r.t. each free logAlpha
        # ∂L/∂(logAlpha_free[k]) = ∂L/∂(logAlpha[k+1]) - ∂L/∂(logAlpha[0])
        gradLogAlphaFree = gradLogAlpha[1:] - gradLogAlpha[0]
        
        grad = np.concatenate([gradLogAlphaFree, gradLogBeta, [gradLogGamma]])
        return negLL, grad
    
    x0 = np.zeros(2 * nTeams)
    x0[2*nTeams - 1] = np.log(1.3)
    
    if verbose:
        print(f"  Optimizing {len(x0)} parameters with analytical gradient...")
    result = minimize(negLogLikAndGrad, x0, jac=True, method='L-BFGS-B',
                      options={'maxiter': 5000, 'maxfun': 100000,
                               'gtol': 1e-6, 'ftol': 1e-10})
    
    logAlphaFree = result.x[:nTeams-1]
    logAlpha0 = -logAlphaFree.sum()
    logAlpha = np.concatenate([[logAlpha0], logAlphaFree])
    logBeta = result.x[nTeams-1:2*nTeams-1]
    logGamma = result.x[2*nTeams-1]
    alpha, beta, gamma = np.exp(logAlpha), np.exp(logBeta), np.exp(logGamma)
    
    if verbose:
        print(f"  Converged: {result.success}, iterations: {result.nit}")
        print(f"  Final negLogLik: {result.fun:,.1f}")
        print(f"  Max gradient component: {abs(result.jac).max():.6f}")
        print(f"  Home advantage (gamma): {gamma:.3f}")
    
    return {
        'attack': {t: alpha[i] for i, t in enumerate(eligible)},
        'defense': {t: beta[i] for i, t in enumerate(eligible)},
        'homeAdv': gamma,
        'teams': eligible,
        'logLik': -result.fun,
        'converged': result.success,
    }


# Fit it
print("=" * 60)
maherFit = fitMaherModel(matches, weights, minMatches=10)
print("=" * 60)

Fitting Maher model on 228 teams (10+ matches)
  Matches used: 7,565 (97.8% of total)
  Optimizing 456 parameters with analytical gradient...


  Converged: True, iterations: 335
  Final negLogLik: 2,612.5
  Max gradient component: 0.043962
  Home advantage (gamma): 1.260


## 4. Inspect the fit

Let's verify the parameters look sensible. Top attacking teams should be Spain, Germany, England etc. Best defenses should be teams like Morocco, Italy, Argentina.

In [5]:
# Build summary DataFrame
summary = pd.DataFrame([
    {'team': t,
     'attack': maherFit['attack'][t],
     'defense': maherFit['defense'][t],
     'attackMinusDefense': maherFit['attack'][t] - maherFit['defense'][t]}
    for t in maherFit['teams']
])

print("Top 15 ATTACK strength (scores most vs avg defense):")
print(summary.nlargest(15, 'attack')[['team', 'attack', 'defense']].to_string(index=False))

print("\nTop 15 DEFENSE (lowest beta = concedes least):")
print(summary.nsmallest(15, 'defense')[['team', 'attack', 'defense']].to_string(index=False))

print("\nTop 15 OVERALL (attack - defense, our proxy for total strength):")
print(summary.nlargest(15, 'attackMinusDefense')[['team', 'attack', 'defense', 'attackMinusDefense']].to_string(index=False))

Top 15 ATTACK strength (scores most vs avg defense):
       team   attack  defense
      Spain 4.585225 0.275899
    Germany 3.905952 0.415544
   Portugal 3.889801 0.334319
     Brazil 3.785742 0.297877
Netherlands 3.772315 0.382229
  Argentina 3.718689 0.213759
     France 3.695844 0.304879
   Colombia 3.646211 0.347193
    England 3.563109 0.257476
     Norway 3.457094 0.416271
    Belgium 3.285061 0.390217
Switzerland 3.204576 0.409720
    Denmark 3.040865 0.381061
    Croatia 2.988268 0.388670
      Italy 2.946983 0.415748

Top 15 DEFENSE (lowest beta = concedes least):
       team   attack  defense
  Argentina 3.718689 0.213759
    Morocco 2.503282 0.249365
    England 3.563109 0.257476
    Ecuador 2.111109 0.258938
      Spain 4.585225 0.275899
    Uruguay 2.445763 0.289338
     Brazil 3.785742 0.297877
     France 3.695844 0.304879
   Portugal 3.889801 0.334319
   Colombia 3.646211 0.347193
       Mali 1.708776 0.356779
    Senegal 2.585442 0.362130
      Japan 2.918948 0.362298

## 5. New scoring function — Maher-based

Replaces the ELO-based `predictScore`. Same signature so it slots into the existing simulator.

In [6]:
def predictScoreMaher(
    teamA: str,
    teamB: str,
    maherFit: dict,
    rng: np.random.Generator,
    homeTeam: Optional[str] = None,  # if A or B is "home" (non-neutral)
) -> tuple[int, int]:
    """Sample a match score using Maher attack/defense parameters.
    
    Args:
        teamA, teamB: team names (use tournament names; gets mapped to dataset names internally)
        maherFit: result of fitMaherModel
        rng: numpy Generator
        homeTeam: if specified and equal to teamA or teamB, applies home advantage to that team.
                  None means neutral venue (default for World Cup matches).
    
    Returns:
        (goalsA, goalsB) integer tuple
    """
    # Map to dataset names
    aData = toDatasetName(teamA)
    bData = toDatasetName(teamB)
    
    alphaA = maherFit['attack'].get(aData)
    betaA = maherFit['defense'].get(aData)
    alphaB = maherFit['attack'].get(bData)
    betaB = maherFit['defense'].get(bData)
    
    if any(p is None for p in [alphaA, betaA, alphaB, betaB]):
        missing = [t for t, p in zip([teamA, teamB], [alphaA, alphaB]) if p is None]
        raise KeyError(f"Maher parameters not found for: {missing}")
    
    gamma = maherFit['homeAdv']
    
    # Apply home advantage
    homeFactorA = gamma if homeTeam == teamA else 1.0
    homeFactorB = gamma if homeTeam == teamB else 1.0
    
    lambdaA = alphaA * betaB * homeFactorA
    lambdaB = alphaB * betaA * homeFactorB
    
    # Sample
    goalsA = int(rng.poisson(np.clip(lambdaA, 0, 20)))
    goalsB = int(rng.poisson(np.clip(lambdaB, 0, 20)))
    return goalsA, goalsB


# Sanity checks
print("Spain vs Curaçao (neutral):", predictScoreMaher('Spain', 'Curaçao', maherFit, rng))
print("Mexico (home) vs Korea Republic:", predictScoreMaher('Mexico', 'Korea Republic', maherFit, rng, homeTeam='Mexico'))
print("Argentina vs France (neutral):", predictScoreMaher('Argentina', 'France', maherFit, rng))
print("Italy-style test — Morocco (defense) vs Senegal:", predictScoreMaher('Morocco', 'Senegal', maherFit, rng))
print()

# Expected goals comparison
print("Expected goals (deterministic):")
for a, b in [('Spain', 'Curaçao'), ('Argentina', 'France'), ('Brazil', 'Morocco'), ('Germany', 'Belgium')]:
    aD, bD = toDatasetName(a), toDatasetName(b)
    lambdaA = maherFit['attack'][aD] * maherFit['defense'][bD]
    lambdaB = maherFit['attack'][bD] * maherFit['defense'][aD]
    print(f"  {a:<12} {lambdaA:.2f} - {lambdaB:.2f} {b}")

Spain vs Curaçao (neutral): (6, 1)
Mexico (home) vs Korea Republic: (1, 3)
Argentina vs France (neutral): (0, 1)
Italy-style test — Morocco (defense) vs Senegal: (2, 0)

Expected goals (deterministic):
  Spain        4.22 - 0.31 Curaçao
  Argentina    1.13 - 0.79 France
  Brazil       0.94 - 0.75 Morocco
  Germany      1.52 - 1.37 Belgium


## 6. Tournament structure

Same as previous version — official 2026 groups.

In [7]:
HOST_NATIONS = {'USA', 'Canada', 'Mexico'}

GROUPS = {
    'A': ['Mexico', 'South Africa', 'Korea Republic', 'Czechia'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', "Côte d'Ivoire", 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

# Verify all teams have Maher parameters
allGroupTeams = list(itertools.chain.from_iterable(GROUPS.values()))
missing = []
for t in allGroupTeams:
    if toDatasetName(t) not in maherFit['attack']:
        missing.append(t)
if missing:
    print(f"⚠ Missing Maher parameters for: {missing}")
else:
    print(f"✓ All 48 teams have fitted Maher parameters")

✓ All 48 teams have fitted Maher parameters


## 7. Knockout match resolution (Maher version)

In [8]:
def simulatePenalties(
    teamA: str,
    teamB: str,
    maherFit: dict,
    rng: np.random.Generator,
) -> tuple[int, int]:
    """Simulate penalty shootout. Skill edge from Maher overall strength (attack - defense)."""
    aData = toDatasetName(teamA)
    bData = toDatasetName(teamB)
    # Skill proxy: log ratio of (attack / defense) — high alpha + low beta means strong
    strA = np.log(maherFit['attack'][aData] / maherFit['defense'][aData])
    strB = np.log(maherFit['attack'][bData] / maherFit['defense'][bData])
    edge = np.clip((strA - strB) * 0.05, -0.05, 0.05)
    pA = 0.75 + edge
    pB = 0.75 - edge
    
    scoreA, scoreB = 0, 0
    for kick in range(5):
        remainingA = 5 - kick
        remainingB = 5 - kick - 1
        if rng.random() < pA: scoreA += 1
        if scoreB + remainingB < scoreA: return scoreA, scoreB
        if rng.random() < pB: scoreB += 1
        if scoreA + remainingA - 1 < scoreB: return scoreA, scoreB
    
    while scoreA == scoreB:
        if rng.random() < pA: scoreA += 1
        if rng.random() < pB: scoreB += 1
    
    return scoreA, scoreB


def predictKnockoutScoreMaher(
    teamA: str,
    teamB: str,
    maherFit: dict,
    rng: np.random.Generator,
    homeTeam: Optional[str] = None,
    etScale: float = 1/3,
) -> dict:
    """Knockout match: 90' → ET → penalties."""
    aData = toDatasetName(teamA)
    bData = toDatasetName(teamB)
    gamma = maherFit['homeAdv']
    
    homeFactorA = gamma if homeTeam == teamA else 1.0
    homeFactorB = gamma if homeTeam == teamB else 1.0
    
    # 90 minutes
    lambdaA90 = maherFit['attack'][aData] * maherFit['defense'][bData] * homeFactorA
    lambdaB90 = maherFit['attack'][bData] * maherFit['defense'][aData] * homeFactorB
    a90 = int(rng.poisson(lambdaA90))
    b90 = int(rng.poisson(lambdaB90))
    
    if a90 != b90:
        return {'goals90': (a90, b90), 'goalsET': None, 'pens': None,
                'winner': 'A' if a90 > b90 else 'B',
                'decidedIn': '90', 'finalScore': (a90, b90)}
    
    # Extra time
    aET = int(rng.poisson(lambdaA90 * etScale))
    bET = int(rng.poisson(lambdaB90 * etScale))
    aTotal = a90 + aET
    bTotal = b90 + bET
    
    if aTotal != bTotal:
        return {'goals90': (a90, b90), 'goalsET': (aET, bET), 'pens': None,
                'winner': 'A' if aTotal > bTotal else 'B',
                'decidedIn': 'ET', 'finalScore': (aTotal, bTotal)}
    
    # Penalties
    penA, penB = simulatePenalties(teamA, teamB, maherFit, rng)
    return {'goals90': (a90, b90), 'goalsET': (aET, bET), 'pens': (penA, penB),
            'winner': 'A' if penA > penB else 'B',
            'decidedIn': 'PEN', 'finalScore': (aTotal, bTotal)}


# Sanity
print("Knockout sample: Spain vs France")
print(predictKnockoutScoreMaher('Spain', 'France', maherFit, rng))

Knockout sample: Spain vs France
{'goals90': (3, 1), 'goalsET': None, 'pens': None, 'winner': 'A', 'decidedIn': '90', 'finalScore': (3, 1)}


## 8. Group + knockout simulation

In [9]:
@dataclass
class TeamRecord:
    name: str
    played: int = 0
    wins: int = 0
    draws: int = 0
    losses: int = 0
    goalsFor: int = 0
    goalsAgainst: int = 0
    
    @property
    def points(self) -> int:
        return self.wins * 3 + self.draws
    
    @property
    def goalDiff(self) -> int:
        return self.goalsFor - self.goalsAgainst
    
    def applyResult(self, scored, conceded):
        self.played += 1
        self.goalsFor += scored
        self.goalsAgainst += conceded
        if scored > conceded: self.wins += 1
        elif scored == conceded: self.draws += 1
        else: self.losses += 1


def simulateGroupMaher(groupName, teamNames, maherFit, hostNations, rng):
    records = {n: TeamRecord(name=n) for n in teamNames}
    matches = []
    for a, b in itertools.combinations(teamNames, 2):
        home = None
        if a in hostNations: home = a
        elif b in hostNations: home = b
        ga, gb = predictScoreMaher(a, b, maherFit, rng, homeTeam=home)
        records[a].applyResult(ga, gb)
        records[b].applyResult(gb, ga)
        matches.append((a, ga, gb, b))
    return records, matches


def rankGroup(records):
    return sorted(records.values(),
                  key=lambda r: (r.points, r.goalDiff, r.goalsFor),
                  reverse=True)


def simulateAllGroupsMaher(groups, maherFit, hostNations, rng):
    results, allMatches = {}, {}
    for gName, teams in groups.items():
        recs, matches = simulateGroupMaher(gName, teams, maherFit, hostNations, rng)
        results[gName] = rankGroup(recs)
        allMatches[gName] = matches
    return results, allMatches


def selectQualifiers(groupResults):
    winners, runnersUp, allThird, eliminated = {}, {}, [], []
    for gName, ranking in groupResults.items():
        winners[gName] = ranking[0]
        runnersUp[gName] = ranking[1]
        allThird.append((gName, ranking[2]))
        eliminated.append(ranking[3])
    rankedThirds = sorted(allThird,
                          key=lambda x: (x[1].points, x[1].goalDiff, x[1].goalsFor),
                          reverse=True)
    qualifyingThirds = rankedThirds[:8]
    for gName, rec in rankedThirds[8:]:
        eliminated.append(rec)
    return {'winners': winners, 'runnersUp': runnersUp,
            'qualifyingThirds': qualifyingThirds, 'eliminated': eliminated}


def buildRoundOf32(qualifiers):
    winners = sorted(qualifiers['winners'].values(),
                     key=lambda r: (r.points, r.goalDiff, r.goalsFor), reverse=True)
    runnersUp = sorted(qualifiers['runnersUp'].values(),
                       key=lambda r: (r.points, r.goalDiff, r.goalsFor), reverse=True)
    thirds = [rec for _, rec in qualifiers['qualifyingThirds']]
    
    matchups = []
    for w, t in zip(winners[:8], reversed(thirds)):
        matchups.append((w, t))
    for w, r in zip(winners[8:12], runnersUp[:4]):
        matchups.append((w, r))
    remaining = runnersUp[4:]
    for i in range(0, len(remaining), 2):
        matchups.append((remaining[i], remaining[i+1]))
    return matchups


def simulateKnockoutRoundMaher(matchups, maherFit, rng, roundName=''):
    winners, results = [], []
    for tA, tB in matchups:
        outcome = predictKnockoutScoreMaher(tA.name, tB.name, maherFit, rng)
        winner = tA if outcome['winner'] == 'A' else tB
        winners.append(winner)
        results.append({'round': roundName, 'teamA': tA.name, 'teamB': tB.name,
                        'score90': outcome['goals90'], 'scoreET': outcome['goalsET'],
                        'pens': outcome['pens'], 'finalScore': outcome['finalScore'],
                        'decidedIn': outcome['decidedIn'], 'winner': winner.name})
    return winners, results


def simulateKnockoutBracketMaher(roundOf32, maherFit, rng):
    allResults = []
    r16Teams, r = simulateKnockoutRoundMaher(roundOf32, maherFit, rng, 'R32'); allResults.extend(r)
    r16M = [(r16Teams[i], r16Teams[i+1]) for i in range(0, 16, 2)]
    qfTeams, r = simulateKnockoutRoundMaher(r16M, maherFit, rng, 'R16'); allResults.extend(r)
    qfM = [(qfTeams[i], qfTeams[i+1]) for i in range(0, 8, 2)]
    sfTeams, r = simulateKnockoutRoundMaher(qfM, maherFit, rng, 'QF'); allResults.extend(r)
    sfM = [(sfTeams[i], sfTeams[i+1]) for i in range(0, 4, 2)]
    finalists, r = simulateKnockoutRoundMaher(sfM, maherFit, rng, 'SF'); allResults.extend(r)
    champs, r = simulateKnockoutRoundMaher([(finalists[0], finalists[1])], maherFit, rng, 'FINAL')
    allResults.extend(r)
    return {'allResults': allResults, 'champion': champs[0],
            'runnerUp': finalists[1] if champs[0].name == finalists[0].name else finalists[0],
            'semifinalists': [t.name for t in sfTeams],
            'quarterfinalists': [t.name for t in qfTeams]}


def simulateTournamentMaher(groups, maherFit, hostNations, rng):
    groupResults, _ = simulateAllGroupsMaher(groups, maherFit, hostNations, rng)
    qualifiers = selectQualifiers(groupResults)
    r32 = buildRoundOf32(qualifiers)
    ko = simulateKnockoutBracketMaher(r32, maherFit, rng)
    return {'groupResults': groupResults, 'qualifiers': qualifiers,
            'knockoutResults': ko['allResults'], 'champion': ko['champion'].name,
            'runnerUp': ko['runnerUp'].name, 'semifinalists': ko['semifinalists'],
            'quarterfinalists': ko['quarterfinalists']}


# Single run
sim = simulateTournamentMaher(GROUPS, maherFit, HOST_NATIONS, np.random.default_rng(42))
print(f"Champion: {sim['champion']}")
print(f"Runner-up: {sim['runnerUp']}")
print(f"Semi-finalists: {sim['semifinalists']}")

Champion: England
Runner-up: Spain
Semi-finalists: ['Argentina', 'England', 'Spain', 'Morocco']


## 9. Monte Carlo

Run 1000 tournaments and aggregate.

In [10]:
def runMonteCarloMaher(nSims, groups, maherFit, hostNations, baseSeed=0):
    champions, runnersUp, sfs, qfs = [], [], [], []
    for i in range(nSims):
        simRng = np.random.default_rng(baseSeed + i)
        result = simulateTournamentMaher(groups, maherFit, hostNations, simRng)
        champions.append(result['champion'])
        runnersUp.append(result['runnerUp'])
        sfs.extend(result['semifinalists'])
        qfs.extend(result['quarterfinalists'])
    
    allTeams = list(itertools.chain.from_iterable(groups.values()))
    summary = []
    for team in allTeams:
        tData = toDatasetName(team)
        summary.append({
            'team': team,
            'attack': maherFit['attack'].get(tData, np.nan),
            'defense': maherFit['defense'].get(tData, np.nan),
            'titleProb': champions.count(team) / nSims,
            'finalProb': (champions.count(team) + runnersUp.count(team)) / nSims,
            'sfProb': sfs.count(team) / nSims,
            'qfProb': qfs.count(team) / nSims,
        })
    return pd.DataFrame(summary).sort_values('titleProb', ascending=False).reset_index(drop=True)


print("Running 1000 simulations...")
mcResults = runMonteCarloMaher(1000, GROUPS, maherFit, HOST_NATIONS, baseSeed=1000)

print("\nTop 20 by title probability:")
print(mcResults.head(20).to_string(index=False))

Running 1000 simulations...



Top 20 by title probability:
       team   attack  defense  titleProb  finalProb  sfProb  qfProb
      Spain 4.585225 0.275899      0.179      0.283   0.407   0.594
  Argentina 3.718689 0.213759      0.169      0.254   0.372   0.551
    England 3.563109 0.257476      0.109      0.181   0.318   0.498
     Brazil 3.785742 0.297877      0.098      0.182   0.308   0.507
   Portugal 3.889801 0.334319      0.075      0.127   0.231   0.379
   Colombia 3.646211 0.347193      0.053      0.097   0.188   0.334
     France 3.695844 0.304879      0.050      0.114   0.231   0.396
    Germany 3.905952 0.415544      0.035      0.082   0.161   0.316
    Morocco 2.503282 0.249365      0.034      0.088   0.171   0.352
Netherlands 3.772315 0.382229      0.030      0.086   0.166   0.331
    Belgium 3.285061 0.390217      0.022      0.055   0.132   0.295
    Croatia 2.988268 0.388670      0.022      0.042   0.118   0.268
     Norway 3.457094 0.416271      0.021      0.048   0.111   0.234
    Uruguay 2.4457

## 10. Validation

Sanity checks and comparison to the ELO-only version.

In [11]:
# Sanity
assert abs(mcResults['titleProb'].sum() - 1.0) < 1e-9
print(f"✓ Title probabilities sum to {mcResults['titleProb'].sum():.4f}")
print(f"✓ Max title prob: {mcResults['titleProb'].max():.3f} ({mcResults.iloc[0]['team']})")
print(f"✓ Teams winning at least once: {(mcResults['titleProb'] > 0).sum()} of 48")
print(f"\nHost nations:")
for host in sorted(HOST_NATIONS):
    row = mcResults[mcResults['team'] == host].iloc[0]
    print(f"  {host}: title={row['titleProb']:.3f}, SF={row['sfProb']:.3f}")

✓ Title probabilities sum to 1.0000
✓ Max title prob: 0.179 (Spain)
✓ Teams winning at least once: 29 of 48

Host nations:
  Canada: title=0.003, SF=0.037
  Mexico: title=0.002, SF=0.044
  USA: title=0.000, SF=0.018


## 11. Notes and next steps

**Where this model is honest about its limits:**
1. **Independence assumption** — the model treats home goals and away goals as independent Poissons. In reality there's mild negative correlation in low-scoring matches (Dixon-Coles tau correction fixes this; worth adding for marginal improvement)
2. **No score-state effects** — once a team is up 3-0 they ease off; the model doesn't capture this
3. **No squad turnover** — Germany's 2018 squad is 70% different from 2026's, but the model assumes "Germany" is a stable entity. Time decay partially mitigates this
4. **Importance weights are heuristic** — academics have studied this; my weights are reasonable but not calibrated

**For further work**:
- **Dixon-Coles correction**: 5-line addition to the likelihood that improves low-score predictions
- **Calibration check**: run model on 2022 WC data only and see if it would have predicted Argentina winning at a reasonable rate
- **Compare to your existing ELO**: pull `elo_ratings.csv` from `calculate_elo.py` output, take final ELO per team, and see if Maher attack-defense correlates well with ELO. They should — strong teams should have high attack AND low defense